# 🔍 Schema Validator

Yeh notebook **2 files** read karke schema validation karta hai:

| File | Description |
|------|-------------|
| **File 1** (Schema) | 3 columns: `id`, `field_name`, `datatype` |
| **File 2** (Data)   | Columns = File 1 ki `field_name` values |

**Kya check hota hai:**
- ✅ Har column ka data type sahi hai ya nahi
- ❌ Data file mein extra columns toh nahi (schema se zyada)
- ⚠️ Schema ke kuch fields data mein missing toh nahi

**Supported data types:** `STRING`, `BIGINT`, `DECIMAL`, `DECIMAL(38,18)`, `INTEGER`, `FLOAT`, `BOOLEAN`, `DATE`, `DATETIME`, `TIMESTAMP`, etc.

**Supported file formats:** `.csv`, `.xlsx`, `.xls`

---
## 📦 Cell 1 — Libraries Install (sirf pehli baar)

In [4]:
# Agar pandas/openpyxl install nahi hain toh yeh run karo
# Pehle se installed hain toh skip kar sakte ho
!pip install pandas openpyxl xlrd -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

---
## ⚙️ Cell 2 — File Paths (Sirf yahan change karo)

In [8]:
# ──────────────────────────────────────────────────────────
#   YEH DO PATHS APNI FILES KE HISAAB SE BADLO
# ──────────────────────────────────────────────────────────

SCHEMA_FILE = "/home/shariq/Downloads/schemafile.csv"   # <-- File 1: schema file ka path
DATA_FILE   = "/home/shariq/Downloads/datafile.csv"     # <-- File 2: data file ka path

# Examples:
# SCHEMA_FILE = "C:/Users/Ali/Documents/schema.xlsx"
# DATA_FILE   = "C:/Users/Ali/Documents/ingested_data.csv"

# ──────────────────────────────────────────────────────────
# Screenshot mein schema ka column 'datatype' tha (data_type nahi)
# Agar tumhari file mein alag naam hai toh yahan badlo:
# ──────────────────────────────────────────────────────────
DATATYPE_COLUMN_NAME = "datatype"   # <-- 'datatype' ya 'data_type' jo bhi schema mein ho

print(f"✅ Schema file      : {SCHEMA_FILE}")
print(f"✅ Data file        : {DATA_FILE}")
print(f"✅ Datatype column  : {DATATYPE_COLUMN_NAME}")

✅ Schema file      : /home/shariq/Downloads/schemafile.csv
✅ Data file        : /home/shariq/Downloads/datafile.csv
✅ Datatype column  : datatype


---
## 📚 Cell 3 — Imports aur Type Mapping

In [9]:
import re
import pandas as pd
import numpy as np
from IPython.display import display

# ─── Base type mapping ────────────────────────────────────────────────────────
TYPE_ALIASES = {
    # Integer / BIGINT
    "int": "int",         "integer": "int",    "int32": "int",
    "int64": "int",       "bigint": "int",     "smallint": "int",   "tinyint": "int",
    # Float / DECIMAL
    "float": "float",     "float32": "float",  "float64": "float",
    "double": "float",    "decimal": "float",  "numeric": "float",
    "real": "float",      "number": "float",
    # String
    "str": "str",         "string": "str",     "varchar": "str",
    "char": "str",        "text": "str",       "nvarchar": "str",   "nchar": "str",
    # Boolean
    "bool": "bool",       "boolean": "bool",
    # Date/Datetime
    "date": "date",       "datetime": "datetime", "timestamp": "datetime", "time": "time",
}


def normalize_dtype(raw: str) -> str:
    """
    Schema datatype string ko base category mein convert karta hai.
    'DECIMAL(38,18)' -> 'float'
    'VARCHAR(255)'   -> 'str'
    'BIGINT'         -> 'int'
    """
    cleaned = str(raw).strip().lower()
    # Parentheses wala part hatao: decimal(38,18) -> decimal
    base = re.sub(r"\(.*?\)", "", cleaned).strip()
    return TYPE_ALIASES.get(base, None)


print("✅ Libraries load ho gayi!")
print()
print("Type normalization test (tumhare screenshot ke types):")
for t in ["DECIMAL(38,18)", "DECIMAL", "BIGINT", "STRING", "INTEGER", "VARCHAR(255)"]:
    print(f"  '{t:20s}' → '{normalize_dtype(t)}'")

✅ Libraries load ho gayi!

Type normalization test (tumhare screenshot ke types):
  'DECIMAL(38,18)      ' → 'float'
  'DECIMAL             ' → 'float'
  'BIGINT              ' → 'int'
  'STRING              ' → 'str'
  'INTEGER             ' → 'int'
  'VARCHAR(255)        ' → 'str'


---
## 🔧 Cell 4 — Helper Functions

In [5]:
def read_file(filepath: str) -> pd.DataFrame:
    """CSV ya Excel file padhta hai — sab values string mein load hoti hain."""
    lower = filepath.lower()
    kwargs = {"dtype": str, "keep_default_na": False}
    if lower.endswith(".csv"):
        return pd.read_csv(filepath, **kwargs)
    elif lower.endswith(".xlsx"):
        return pd.read_excel(filepath, engine="openpyxl", dtype=str)
    elif lower.endswith(".xls"):
        return pd.read_excel(filepath, engine="xlrd", dtype=str)
    else:
        try:
            return pd.read_csv(filepath, **kwargs)
        except Exception:
            raise ValueError(f"Unsupported format: '{filepath}'. Use .csv / .xlsx / .xls")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Column names se whitespace hatao aur lowercase karo."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def is_empty(value) -> bool:
    """Value empty/null hai?"""
    if value is None:
        return True
    s = str(value).strip()
    return s == "" or s.lower() in ("nan", "none", "null", "na", "n/a")


def check_value_type(value, expected_type: str) -> bool:
    """
    Value ka type check karta hai.
    Empty/null values ko valid maanta hai.
    """
    if is_empty(value):
        return True

    val_str = str(value).strip()

    if expected_type == "int":
        try:
            f = float(val_str)
            return f == int(f)   # 42.0 valid hai, 42.5 nahi
        except (ValueError, TypeError):
            return False

    elif expected_type == "float":
        try:
            float(val_str)       # scientific notation bhi handle karta hai
            return True
        except (ValueError, TypeError):
            return False

    elif expected_type == "str":
        return True              # STRING mein koi bhi value valid hai

    elif expected_type == "bool":
        return val_str.lower() in ("true", "false", "1", "0", "yes", "no")

    elif expected_type in ("date", "datetime", "time"):
        try:
            pd.to_datetime(val_str)
            return True
        except Exception:
            return False

    return True


print("✅ Helper functions ready hain!")

✅ Helper functions ready hain!


---
## 📂 Cell 5 — Files Load karo aur Preview dekho

In [10]:
# ── Files load karo ──────────────────────────────────────────────────────────
try:
    schema_df = normalize_columns(read_file(SCHEMA_FILE))
    data_df   = normalize_columns(read_file(DATA_FILE))
    print("✅ Dono files successfully load ho gayi!\n")
except FileNotFoundError as e:
    print(f"❌ FILE NOT FOUND: {e}")
    print("   Upar Cell 2 mein file path check karo.")
    raise
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# ── Schema structure check ────────────────────────────────────────────────────
dt_col = DATATYPE_COLUMN_NAME.strip().lower()
required_cols = {"id", "field_name", dt_col}
actual_cols   = set(schema_df.columns)
missing_req   = required_cols - actual_cols

if missing_req:
    print(f"❌ Schema file mein required columns nahi hain: {missing_req}")
    print(f"   Jo columns hain: {sorted(actual_cols)}")
    print(f"   Cell 2 mein DATATYPE_COLUMN_NAME check karo.")
    raise ValueError(f"Missing columns: {missing_req}")

print(f"✅ Schema columns: {sorted(actual_cols)}")
print(f"✅ Data columns  : {len(data_df.columns)} columns found\n")

# ── Previews ─────────────────────────────────────────────────────────────────
print("=" * 55)
print("📄 SCHEMA FILE — Pehli 10 rows")
print("=" * 55)
display(schema_df.head(10))

print(f"\n📊 DATA FILE — {data_df.shape[0]} rows × {data_df.shape[1]} columns")
print("=" * 55)
print("📊 DATA FILE — Pehli 5 rows")
print("=" * 55)
display(data_df.head())

✅ Dono files successfully load ho gayi!

✅ Schema columns: ['datatype', 'field_name', 'id']
✅ Data columns  : 102 columns found

📄 SCHEMA FILE — Pehli 10 rows


,id,field_name,datatype
0,1,CompleteQty,DECIMAL
1,2,CompleteLeaveQty,DECIMAL
2,3,CompleteCumQty,DECIMAL
3,4,CompleteLastShares,DECIMAL
4,5,CompleteAvgPx,DECIMAL
5,6,PercentExecuted,DECIMAL
6,7,ABCD,BIGINT
7,8,Account,STRING
8,9,AvgPx,"DECIMAL(38,18)"
9,10,BoothID,STRING



📊 DATA FILE — 41090 rows × 102 columns
📊 DATA FILE — Pehli 5 rows


,completeleaveqty,s3_path,complianceid,origclordid,lastpx,symbolwithoutsfx,id,changedshares,pad2,hdrorderid,...,orderdate,status,contrabroker,key,account,maxfloor,boothident,symbolsfx,destinationdesc,liquidityindtype
0,0.00000000,s3a://kuza-ignite/bronze/UpdateDAS/NY4/Primary...,,,0.49710000,SELX,KIC-113363214-3,-107,,113363214,...,2026-04-06,,,KIC-113363214-3,KISVG,,,,,
1,0.00000000,s3a://kuza-ignite/bronze/UpdateDAS/NY4/Primary...,,,4.94000000,EOSE,KIC-113362413-3,-5,,113362413,...,2026-04-06,,,KIC-113362413-3,KISVG,,,,,
2,0.00000000,s3a://kuza-ignite/bronze/UpdateDAS/NY4/Primary...,,,55.61000000,JEPQ,KIC-113363172-3,-4,,113363172,...,2026-04-06,,,KIC-113363172-3,KISVG,,,,,
3,0.00000000,s3a://kuza-ignite/bronze/UpdateDAS/NY4/Primary...,LSR59340002589,,1.33000000,AUID,LSS-2226654-3,-1,,2226654,...,2026-04-06,2026-04-06T04:00:09.102,,LSS-2226654-3,LSSC-LSS-600101-D,,,,,
4,0.00000000,s3a://kuza-ignite/bronze/UpdateDAS/NY4/Primary...,,,0.94860000,JZXN,KIC-113363226-3,-49,,113363226,...,2026-04-06,,,KIC-113363226-3,KISVG,,,,,


---
## ✅ Cell 6 — Schema Validation Chalao

In [11]:
dt_col = DATATYPE_COLUMN_NAME.strip().lower()

# ── Schema info extract ───────────────────────────────────────────────────────
schema_fields    = schema_df["field_name"].str.strip().str.lower().tolist()
schema_raw_types = schema_df[dt_col].str.strip().tolist()   # e.g. 'DECIMAL(38,18)', 'BIGINT'
field_type_map   = dict(zip(schema_fields, schema_raw_types))

data_columns = list(data_df.columns)

print("=" * 60)
print("  SCHEMA VALIDATOR")
print("=" * 60)
print(f"  Schema fields  : {len(schema_fields)}")
print(f"  Data columns   : {len(data_columns)}")
print(f"  Data rows      : {len(data_df)}")
print("=" * 60)

# ── Extra columns check ───────────────────────────────────────────────────────
extra_in_data   = [c for c in data_columns if c not in schema_fields]
missing_in_data = [f for f in schema_fields if f not in data_columns]

if extra_in_data:
    print(f"\n❌ ERROR: Data file mein {len(extra_in_data)} extra column(s) hain jo schema mein nahi!")
    print(f"   Extra: {extra_in_data}")
    print("   Data file ya schema fix karo.")
    raise ValueError(f"Extra columns: {extra_in_data}")

if missing_in_data:
    print(f"\n⚠️  WARNING: Schema mein hain lekin data mein nahi: {missing_in_data}")

# ── Type validation ───────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("  DATA TYPE VALIDATION")
print("-" * 60)

common_fields  = [f for f in schema_fields if f in data_columns]
total_errors   = 0
column_results = {}
all_error_rows = []

for field in common_fields:
    raw_dtype   = field_type_map[field]       # 'DECIMAL(38,18)'
    mapped_type = normalize_dtype(raw_dtype)  # 'float'

    if mapped_type is None:
        print(f"\n  ⚠️  '{field}': Unknown type '{raw_dtype}' — skipped")
        column_results[field] = {"status": "skipped", "errors": [], "raw_type": raw_dtype}
        continue

    errors = []
    for row_idx, value in enumerate(data_df[field], start=2):
        if not check_value_type(value, mapped_type):
            errors.append({
                "Row"          : row_idx,
                "Column"       : field,
                "Schema Type"  : raw_dtype,
                "Bad Value"    : value,
                "Python Type"  : type(value).__name__
            })

    if errors:
        total_errors += len(errors)
        column_results[field] = {"status": "fail", "errors": errors, "raw_type": raw_dtype}
        all_error_rows.extend(errors)
        print(f"\n  ❌ '{field}' (schema: {raw_dtype}) — {len(errors)} error(s)")
        for e in errors[:5]:
            print(f"     Row {e['Row']}: '{e['Bad Value']}'")
        if len(errors) > 5:
            print(f"     ... aur {len(errors)-5} errors. Cell 7 mein dekho.")
    else:
        column_results[field] = {"status": "pass", "errors": [], "raw_type": raw_dtype}
        print(f"  ✅ '{field}' (schema: {raw_dtype}) — OK")

# ── Summary ───────────────────────────────────────────────────────────────────
passed  = sum(1 for v in column_results.values() if v["status"] == "pass")
failed  = sum(1 for v in column_results.values() if v["status"] == "fail")
skipped = sum(1 for v in column_results.values() if v["status"] == "skipped")

print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
print(f"  ✅ Passed  : {passed}")
print(f"  ❌ Failed  : {failed}")
print(f"  ⚠️  Skipped : {skipped}")
print(f"  Total type errors: {total_errors}")
if missing_in_data:
    print(f"  ⚠️  Missing in data: {missing_in_data}")
print()
if total_errors == 0 and not missing_in_data:
    print("  🎉 Sab theek hai! Schema sahi apply hua hai.")
else:
    print("  ⚠️  Validation failed. Neeche error details dekho.")
print("=" * 60)

  SCHEMA VALIDATOR
  Schema fields  : 102
  Data columns   : 102
  Data rows      : 41090

------------------------------------------------------------
  DATA TYPE VALIDATION
------------------------------------------------------------
  ✅ 'completeqty' (schema: DECIMAL) — OK
  ✅ 'completeleaveqty' (schema: DECIMAL) — OK
  ✅ 'completecumqty' (schema: DECIMAL) — OK
  ✅ 'completelastshares' (schema: DECIMAL) — OK
  ✅ 'completeavgpx' (schema: DECIMAL) — OK
  ✅ 'percentexecuted' (schema: DECIMAL) — OK
  ✅ 'abcd' (schema: BIGINT) — OK
  ✅ 'account' (schema: STRING) — OK
  ✅ 'avgpx' (schema: DECIMAL(38,18)) — OK
  ✅ 'boothid' (schema: STRING) — OK
  ✅ 'byteorder' (schema: STRING) — OK
  ✅ 'cache' (schema: STRING) — OK
  ✅ 'changedshares' (schema: BIGINT) — OK
  ✅ 'clordid' (schema: STRING) — OK
  ✅ 'clientid' (schema: STRING) — OK
  ✅ 'complianceid' (schema: STRING) — OK
  ✅ 'contrabroker' (schema: STRING) — OK
  ✅ 'cumqty' (schema: BIGINT) — OK
  ✅ 'destination' (schema: STRING) — OK
  ✅ 'e

---
## 📋 Cell 7 — Error Detail Table

In [13]:
if all_error_rows:
    error_df = pd.DataFrame(all_error_rows)
    print(f"❌ Total {len(error_df)} type errors mili hain:\n")
    display(error_df)
else:
    print("✅ Koi type error nahi mili! Data clean hai.")

✅ Koi type error nahi mili! Data clean hai.


---
## 📊 Cell 8 — Column-wise Summary Table

In [14]:
summary_rows = []
for field, info in column_results.items():
    mapped = normalize_dtype(info["raw_type"])
    emoji  = {"pass": "✅", "fail": "❌", "skipped": "⚠️"}.get(info["status"], "?")
    summary_rows.append({
        "Column"          : field,
        "Schema Type"     : info["raw_type"],
        "Mapped Base Type": mapped or "unknown",
        "Status"          : f"{emoji} {info['status'].upper()}",
        "Error Count"     : len(info["errors"]),
    })

summary_df = pd.DataFrame(summary_rows)
print("Column-wise Validation Summary:")
display(summary_df)

Column-wise Validation Summary:


,Column,Schema Type,Mapped Base Type,Status,Error Count
0,completeqty,DECIMAL,float,✅ PASS,0
1,completeleaveqty,DECIMAL,float,✅ PASS,0
2,completecumqty,DECIMAL,float,✅ PASS,0
3,completelastshares,DECIMAL,float,✅ PASS,0
4,completeavgpx,DECIMAL,float,✅ PASS,0
...,...,...,...,...,...
97,symbolsfx,STRING,str,✅ PASS,0
98,hdrboothid,ARRAY<STRING>,unknown,⚠️ SKIPPED,0
99,hdrsymbol,ARRAY<STRING>,unknown,⚠️ SKIPPED,0
100,inet_address,ARRAY<STRING>,unknown,⚠️ SKIPPED,0
